In [1]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Teguh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [3]:
import torchtext; torchtext.disable_torchtext_deprecation_warning()
from torchtext.data.utils import get_tokenizer
import os
from dataset import clean_data, create_text_data, split_data, get_transforms
from inference import inference
from PIL import Image
from model import EncoderCNN
import torch

In [ ]:
# init directory and file path
root_path = '../'
data_root_path = f"{root_path}dataset"
text_dataset = f"{data_root_path}/text/Flickr8k.token.bahasa.txt"
clean_text_dataset = f"{data_root_path}/text/Flickr8k.token.bahasa.clean.txt"

train_file = data_root_path + "/text/" + "Flickr_8k.trainImages.txt"
val_file = data_root_path +  "/text/" + "Flickr_8k.devImages.txt"
test_file = data_root_path + "/text/" + "Flickr_8k.testImages.txt"

# init tokenizer
tokenizer = get_tokenizer('basic_english')

# Clean dataset
if not os.path.exists(clean_text_dataset):
    clean_data(text_dataset, clean_text_dataset, data_root_path)
    
dataframe, vocab = create_text_data(clean_text_dataset, tokenizer=tokenizer)

# split data train, val, test
_, _, test_df = split_data(dataframe, train_file, val_file, test_file)

In [ ]:
d_model = 512
n_head = 4
cnn_size = 'm'
model_path = f'saved_model/{d_model}_{n_head}_{cnn_size}.pth'
image_size = 288
transforms = get_transforms(image_size)
cnn_backbone = 'efficientnetv2_rw_s.ra2_in1k'
cnn_backbone = 'efficientnetv2_rw_m.agc_in1k'

In [ ]:
# # Setup
# batch_size = 32  # or any other size suitable for your hardware
# data = {
#     'filename': [],
#     'predicted': [],
#     'target': []
# }

# # Process in batches
# for i in range(0, len(test_df), batch_size):
#     batch_filenames = test_df['image_filename'][i:i+batch_size]
#     batch_captions = test_df['captions'][i:i+batch_size]
#     batch_images = [Image.open(f"{data_root_path}/images/{filename}").convert('RGB') for filename in batch_filenames]

#     # Run batch inference
#     batch_predictions = batch_inference(batch_images, model_path, d_model, n_head, vocab)

#     # Process each prediction in the batch
#     for filename, captions, prediction_words in zip(batch_filenames, batch_captions, batch_predictions):
#         predicted = ' '.join(word for word in prediction_words if word != '<end>')
#         data['filename'].append(filename)
#         data['predicted'].append(predicted)
#         data['target'].append(captions)

In [ ]:
print(cnn_backbone)

efficientnetv2_rw_m.agc_in1k


In [ ]:
import pandas as pd
from tqdm import tqdm

data = {
    'filename': [],
    'predicted': [],
    'target': []
}
start = 1
end = 20
# Use tqdm to wrap your iterable
for image_filename, captions in tqdm(zip(test_df['image_filename'][start:end], test_df['captions'][start:end]), total=end-start):
    image = Image.open(f"{data_root_path}/images/{image_filename}").convert('RGB')
    
    words = inference(image, model_path, d_model, n_head, vocab, cnn_backbone)
    predicted = ' '.join(word for word in words if word != '<end>')
    new_row = {
        'filename' : image_filename,
        'predicted': predicted,
        'target': captions  # Assuming you want to collect captions as target
    }
    
    for key, value in new_row.items():
        data[key].append(value)
    
data_df = pd.DataFrame(data)
data_df.to_csv(f'testing_greedy_{start}_{end}_{cnn_size}.csv', sep=",", index_label='index')

In [7]:
import pandas as pd
from tqdm import tqdm

data = {
    'filename': [],
    'predicted': [],
    'target': []
}
start = 500
end = 750
# Use tqdm to wrap your iterable
for image_filename, captions in tqdm(zip(test_df['image_filename'][start:end], test_df['captions'][start:end]), total=end-start):
    image = Image.open(f"{data_root_path}/images/{image_filename}").convert('RGB')
    
    words = inference(image, model_path, d_model, n_head, vocab, cnn_backbone)
    predicted = ' '.join(word for word in words if word != '<end>')
    new_row = {
        'filename' : image_filename,
        'predicted': predicted,
        'target': captions  # Assuming you want to collect captions as target
    }
    
    for key, value in new_row.items():
        data[key].append(value)
    
data_df = pd.DataFrame(data)
data_df.to_csv(f'testing_results/testing_{start}_{end}_{cnn_size}.csv', sep=",", index_label='index')

  0%|          | 0/250 [00:00<?, ?it/s]e:\venvs\gtorch\lib\site-packages\torch\nn\modules\conv.py:456: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\cudnn\Conv_v8.cpp:919.)
  return F.conv2d(input, weight, bias, self.stride,
e:\venvs\gtorch\lib\site-packages\torch\nn\modules\activation.py:1221: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  return torch._native_multi_head_attention(
100%|██████████| 250/250 [58:44<00:00, 14.10s/it] 
